In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math

# โมเดล CNN แบบง่าย
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# ข้อมูล MNIST
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# พารามิเตอร์สำหรับ Warm-up และ Decay
total_epochs = 20
warmup_epochs = 5
lr_start = 0.0
lr_max = 0.1
lr_end = 0.001
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr_max)

# สร้าง scheduler แบบ custom
def lr_lambda(current_epoch):
    if current_epoch < warmup_epochs:
        return lr_start + (lr_max - lr_start) * (current_epoch / warmup_epochs)
    else:
        decay_epochs = total_epochs - warmup_epochs
        decay_progress = (current_epoch - warmup_epochs) / decay_epochs
        return lr_max - (lr_max - lr_end) * decay_progress

# สร้าง scheduler โดยอิงจาก epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: lr_lambda(epoch))




In [ ]:
# ฝึกโมเดล
criterion = nn.CrossEntropyLoss()

for epoch in range(total_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1}/{total_epochs} | LR: {current_lr:.6f} | Loss: {running_loss:.4f}")

Epoch 1/20 | LR: 0.002000 | Loss: 2159.7222
Epoch 2/20 | LR: 0.004000 | Loss: 105.6232
Epoch 3/20 | LR: 0.006000 | Loss: 50.1183
Epoch 4/20 | LR: 0.008000 | Loss: 43.7669
Epoch 5/20 | LR: 0.010000 | Loss: 48.5584
Epoch 6/20 | LR: 0.009340 | Loss: 53.4687
Epoch 7/20 | LR: 0.008680 | Loss: 40.0740
Epoch 8/20 | LR: 0.008020 | Loss: 37.9518
Epoch 9/20 | LR: 0.007360 | Loss: 32.4723
Epoch 10/20 | LR: 0.006700 | Loss: 24.1552
Epoch 11/20 | LR: 0.006040 | Loss: 19.0926
Epoch 12/20 | LR: 0.005380 | Loss: 13.8010
Epoch 13/20 | LR: 0.004720 | Loss: 11.9995


KeyboardInterrupt: 